# A magpylib scene in Jupyter

The same 3D view as `marimo_demo.py` — the studio's three.js renderer, handed
the same payload, wrapped as an `anywidget` — in a Jupyter notebook.

Run it with this repo's `.venv` as the kernel. In VS Code, open the notebook and
pick that interpreter; for JupyterLab, `uv pip install jupyterlab` into the same
environment and `jupyter lab examples/jupyter_demo.ipynb`.

Put the pointer on a view and its tools show in the top-right corner: the
legend, the axes, framing, the projection, the theme (auto, light, dark),
export, full screen. Click a view
and it takes the keys the studio's panel uses — **F** frames the selection,
**Home** everything, **1** **3** **7** look from the front, the right and the
top, **5** switches the projection, **H** hides the selection and **shift-H**
shows only it, **Esc** lets go, **space** plays a run.

In [ ]:
import ipywidgets as widgets
import magpylib as magpy
import numpy as np

from magpylib_studio.widget import SceneWidget

## Sliders that rebuild the scene

Jupyter does not re-run a cell when a slider moves, so the sliders say what to
do themselves: rebuild the rings and hand them to the view with `update()`.
The view is made once and re-pointed rather than remade, which is what keeps the
camera where you left it — zoom in and drag a slider, the zoom stays.

The legend over the view is the scene as it is nested: the stack, its rings,
their magnets. Fold a ring with its caret, hide it with the eye at the end of
its row, click a row to select everything under it (⌘/ctrl-click adds,
shift-click takes a range), double-click to frame it. What you select is
`scene.selected` in Python, and `scene.picked` turns it back into the magpylib
objects — the readout under the view follows it.

In [ ]:
def build(count, radius, tilt):
    """Two rings of magnets, turned opposite ways, and a probe in the middle."""

    def ring(name, z, turn):
        ring = magpy.Collection(style_label=f"{name} ring")
        for i in range(count):
            magnet = magpy.magnet.Cuboid(
                dimension=(1, 1, 1),
                polarization=(0, 0, 1),
                position=(radius, 0, z),
                style_label=f"{name} {i + 1}",
            )
            # Turned in place by the same angle for every magnet -- 0 degrees
            # is an axial ring, 90 a radial one -- then carried round.
            magnet.rotate_from_angax(turn, "y", anchor=magnet.position)
            magnet.rotate_from_angax(360 * i / count, "z", anchor=(0, 0, 0))
            ring.add(magnet)
        return ring

    stack = magpy.Collection(
        ring("upper", 1.0, tilt), ring("lower", -1.0, -tilt), style_label="stack"
    )
    return stack, magpy.Sensor(style_label="centre probe")

In [ ]:
count = widgets.IntSlider(6, min=2, max=16, description="magnets")
radius = widgets.FloatSlider(4.0, min=2.0, max=8.0, step=0.5, description="ring radius")
tilt = widgets.IntSlider(45, min=0, max=180, step=15, description="tilt °")
readout = widgets.HTML()
scene = SceneWidget(height=460)
current = {}


def say():
    """The field at the probe, and what is selected, under the view."""
    field = np.round(magpy.getB(current["stack"], current["probe"]) * 1000, 3)
    picked = ", ".join(obj.style.label for obj in scene.picked) or "<i>nothing</i>"
    readout.value = f"<b>B at the probe:</b> {field} mT<br><b>Selected:</b> {picked}"


def rebuild(_=None):
    current["stack"], current["probe"] = build(count.value, radius.value, tilt.value)
    scene.update(current["stack"], current["probe"])
    say()


for slider in (count, radius, tilt):
    slider.observe(rebuild, names="value")
scene.observe(lambda _: say(), names="selected")
rebuild()

widgets.VBox([widgets.HBox([count, radius, tilt]), scene, readout])

## Paths, played in the browser

`animation=True` captures every step of the run. The widget keeps the frames
and serves them to the view one at a time, so **▶** plays and the slider scrubs —
including what a pose cannot express: the probe's field arrows are recomputed
per frame, so they turn as it passes.

In [ ]:
rotor = magpy.magnet.Cylinder(
    polarization=(1, 0, 0), dimension=(4, 1), style_label="rotor"
)
sweep = magpy.Sensor(
    position=np.linspace((-6, 0, 1.5), (6, 0, 1.5), 60), style_label="sweep"
)
sweep.style.arrows.x.show = True
sweep.style.arrows.y.show = True

SceneWidget(rotor, sweep, animation=True, height=380)

## Keeping a view

The export icon saves a view as one HTML file that opens anywhere with no
notebook behind it — the widget itself, legend and keys and all, and a captured
run still plays in it. From a cell, `write_html` does the same, as Plotly's does.

In [ ]:
# scene.write_html("rings.html")

In [ ]:
1 + 1